# B: rewriter (Llama 3.1 8B Instruct, 4-bit)

Run the setup cell first. Needs a GPU runtime (Colab: Runtime > Change runtime type > T4 GPU) and
access to the gated Llama repo with `HF_TOKEN` in Secrets. There is no fallback model: if Llama cannot
be loaded, `get_model()` stops with the reason and the fix (no token, licence not accepted, out of memory).

1. Load the model and time one generation.
2. Run `rewrite()` on the israel_palestine rows of `data/test_sentences.csv`.
3. Report JSON validity, new numbers/names, retries, speed and bias score before/after.

In [ ]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:  # autoreload is broken on some Colab images (Python 3.13: no 'imp' module)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload not available: restart the runtime after editing src/")

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

In [ ]:
import json, re, time
import pandas as pd
import torch
from IPython.display import display

from src import rewriter
from src.detector import detect, explain
from src.rag import retrieve

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
assert not config.USE_FAKES, "DEBIAS_FAKE=1 is set, so rewrite() would use the fake. Unset it."
CONFLICT = "israel_palestine"
DETECTOR_FAKE = config.USE_FAKES or not config.DETECTOR_PATH
print("detector:", "FAKE (scores are placeholders)" if DETECTOR_FAKE else config.DETECTOR_PATH)
print("prompts:", rewriter.prompt_versions())

## 1. Load the model and time one generation

In [ ]:
t0 = time.perf_counter()
model, tokenizer, MODEL_ID = rewriter.get_model()
load_s = time.perf_counter() - t0
print(f"{MODEL_ID} loaded in {load_s:.0f}s, GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")

sample = "Side A troops stormed the village and terrorized residents, local officials said."
msgs = rewriter.build_messages(sample, ["stormed", "terrorized"], retrieve(sample, config.RAG_K))
rewriter.generate_detailed([msgs])  # warm-up, not timed

torch.cuda.synchronize(); t0 = time.perf_counter()
g = rewriter.generate_detailed([msgs])[0]
torch.cuda.synchronize(); gen_s = time.perf_counter() - t0
print(g["text"])
speed = {
    "load_seconds": round(load_s, 1),
    "generation_seconds": round(gen_s, 2),
    "new_tokens": g["new_tokens"],
    "tokens_per_second": round(g["new_tokens"] / gen_s, 1),
    "hit_token_limit": g["hit_limit"],
    "gpu_memory_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
}
print(speed)
utils.save_result("B_rewriter_speed", speed,
                  extra={"model_id": MODEL_ID, "max_new_tokens": config.MAX_NEW_TOKENS, "seed": config.SEED})

## 2. Rewrite the israel_palestine test sentences

Which words are flagged (`SPANS_FROM`):
- `"reference"` (default): the words Amen removed or replaced when writing the `neutral` version
  (`rewriter.spans_from_reference`). This tests the rewriter on its own, with correct flagged words,
  so all sentences reach the model.
- `"detector"`: `src.detector.explain()`, as in the real pipeline (fake until the trained detector is loaded,
  which only flags a few words, so most sentences are skipped).

A `spans` / `flagged` / `flagged_words` / `flagged_spans` column in `test_sentences.csv` overrides both.
Rules come from `src.rag.retrieve`.

In [ ]:
df = pd.read_csv(config.TEST_SENTENCES_PATH)
rows = df[df["conflict"] == CONFLICT].reset_index(drop=True)
if rows.empty:
    raise ValueError(f"{config.TEST_SENTENCES_PATH.name} has no {CONFLICT} rows yet (ask Amen).")

SPANS_FROM = "reference"  # "reference" (rewriter on its own) or "detector" (as in the pipeline)
SPAN_COL = next((c for c in ("spans", "flagged", "flagged_words", "flagged_spans") if c in rows.columns), None)
if SPAN_COL:
    SPAN_SOURCE = f"csv:{SPAN_COL}"
elif SPANS_FROM == "reference":
    SPAN_SOURCE = "reference (biased vs neutral)"
else:
    SPAN_SOURCE = "detector.explain" + (" (fake)" if DETECTOR_FAKE else "")
print(len(rows), "rows | spans from:", SPAN_SOURCE)


def parse_spans(value) -> list[str]:
    if pd.isna(value) or not str(value).strip():
        return []
    value = str(value).strip()
    if value.startswith("["):
        return [str(s) for s in json.loads(value)]
    return [s.strip() for s in re.split(r"[;|]", value) if s.strip()]

In [ ]:
RULE_ID = re.compile(r"\bR\d{2}\b")
records = []
for r in rows.itertuples(index=False):
    text = r.biased
    if SPAN_COL:
        spans = parse_spans(getattr(r, SPAN_COL))
    elif SPANS_FROM == "reference":
        spans = rewriter.spans_from_reference(text, r.neutral)
    else:
        spans = explain(text, CONFLICT)
    rules = retrieve(" ".join(spans) if spans else text, config.RAG_K)

    t0 = time.perf_counter()
    result, info = rewriter.rewrite_detailed(text, spans, rules)
    secs = time.perf_counter() - t0
    contract.validate_rewrite(result)

    attempts = info["attempts"]
    records.append({
        "id": r.id,
        "biased": text,
        "revised": result["revised"],
        "neutral (reference)": r.neutral,
        "spans": spans,
        "changes": "\n".join(f'"{c["original"]}" -> "{c["revised"]}" ({c["reason"]})' for c in result["changes"]),
        "n_changes": len(result["changes"]),
        "rule_ids": sorted({i for c in result["changes"] for i in RULE_ID.findall(c["reason"])}),
        "expected_rules": r.expected_rules,
        "skipped": info["skipped"],
        "json_ok_first_try": bool(attempts) and attempts[0]["parsed"],
        "json_ok_any_try": any(a["parsed"] for a in attempts),
        "retries": info["retries"],
        "fell_back": info["fell_back"],
        "new_numbers_or_names": any(a["new_numbers"] or a["new_names"] for a in attempts),
        "hit_token_limit": any(a["hit_limit"] for a in attempts),
        "reasons_have_rule_id": all(RULE_ID.search(c["reason"]) for c in result["changes"]),
        "seconds": secs,
        "score_before": detect(text, CONFLICT)["score"],
        "score_after": detect(result["revised"], CONFLICT)["score"],
        "errors": [a["errors"] for a in attempts],
    })

res = pd.DataFrame(records)
res.to_csv(ARTIFACTS / "B_rewriter_israel_palestine_rows.csv", index=False)
print("per-row results:", ARTIFACTS / "B_rewriter_israel_palestine_rows.csv")

## 3. Report

Percentages are over sentences that reached the model (non-empty spans).
`pct_new_numbers_or_names` counts a sentence if any attempt (first or retry) added one;
those outputs were rejected, never returned.

In [ ]:
ran = res[~res["skipped"]]
pct = lambda s: round(100 * s.mean(), 1) if len(s) else None
metrics = {
    "n_sentences": len(res),
    "n_sent_to_model": len(ran),
    "n_skipped_no_spans": int(res["skipped"].sum()),
    "pct_valid_json_first_try": pct(ran["json_ok_first_try"]),
    "pct_valid_json_any_try": pct(ran["json_ok_any_try"]),
    "pct_passed_validation": pct(~ran["fell_back"]),
    "pct_new_numbers_or_names": pct(ran["new_numbers_or_names"]),
    "retries_used": int(ran["retries"].sum()),
    "fallbacks_unchanged": int(ran["fell_back"].sum()),
    "pct_hit_token_limit": pct(ran["hit_token_limit"]),
    "pct_reasons_with_rule_id": pct(ran.loc[ran["n_changes"] > 0, "reasons_have_rule_id"]),
    "seconds_per_sentence_mean": round(ran["seconds"].mean(), 2) if len(ran) else None,
    "seconds_per_sentence_median": round(ran["seconds"].median(), 2) if len(ran) else None,
    "bias_score_before_mean": round(res["score_before"].mean(), 3),
    "bias_score_after_mean": round(res["score_after"].mean(), 3),
}
for k, v in metrics.items():
    print(f"{k:32s} {v}")
if DETECTOR_FAKE:
    print("\nNOTE: detector is fake, so bias scores before/after are placeholders.")

utils.save_result(
    "B_rewriter_israel_palestine", metrics,
    extra={"model_id": MODEL_ID, "prompts": rewriter.prompt_versions(), "span_source": SPAN_SOURCE,
           "detector_fake": DETECTOR_FAKE, "rag_k": config.RAG_K,
           "max_new_tokens": config.MAX_NEW_TOKENS, "seed": config.SEED},
)

### 5 examples side by side (changed ones first)

In [ ]:
examples = pd.concat([res[res["n_changes"] > 0], res[res["n_changes"] == 0]]).head(5)
cols = ["id", "biased", "revised", "neutral (reference)", "changes", "score_before", "score_after"]
display(examples[cols].style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"}).hide(axis="index"))

### Failures (for prompt fixes)

In [ ]:
display(res.loc[res["fell_back"], ["id", "biased", "spans", "errors"]])